
# Functional Brain Connectivity Graphs in PPU Adolescents (End-to-End Colab)

Notebook ini membangun ulang pipeline riset "Analisis Graf-Konektivitas Fungsional Otak pada Remaja dengan Penggunaan Pornografi Bermasalah". Seluruh langkah didesain untuk dieksekusi di Google Colab (filesystem utama `/content`).

**Ringkasan tahapan utama:**
1. Setup, konfigurasi, dan pemasangan dependensi.
2. Unduh dataset Kaggle (EEG pornography addiction) secara otomatis.
3. Loading sinyal EEG 19-channel (250 Hz), pemetaan kanal 10–20, dan label subjek (S1–S7 PPU=1, S8–S14 Control=0).
4. Preprocessing: bandpass 1–50 Hz, notch 50 Hz, artefact rejection ±100 µV, epoch 2 detik, dan bandpass alpha/beta.
5. Konektivitas: Coherence & Phase-Lag Index (PLI) per band dan task (EC, ET).
6. Konstruksi graf (weighted & binary threshold 25th percentile) dan metrik teori graf + uji Mann–Whitney U.
7. Dataset PyTorch Geometric + GCN (LOOCV, Optuna untuk hyperparameter tuning, anti-leakage scaling & threshold).
8. Interpretabilitas: ablation frontal, embedding PCA, identifikasi edge kritis.
9. Ekspor semua output ke `/content/outputs/` lalu zip untuk diunduh.


In [ ]:

import os, json, random, math, time, zipfile, shutil, itertools, tempfile, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import scipy.signal as signal
from scipy.stats import mannwhitneyu
import mne
import networkx as nx
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')

CONFIG = {
    "random_seed": 42,
    "sampling_rate": 250,
    "channels_order": [
        "Fp1", "Fp2", "F7", "F3", "Fz", "F4", "F8", "T7", "C3", "Cz", "C4", "T8", "P7", "P3", "Pz", "P4", "P8", "O1", "O2"
    ],
    "bandpass": [1, 50],
    "notch": 50,
    "epoch_length_sec": 2,
    "epoch_overlap": 0.5,
    "artifact_threshold_uV": 100,
    "bands": {"alpha": [8, 13], "beta": [13, 30]},
    "connectivity_metrics": ["coherence", "pli"],
    "threshold_percentile": 25,
    "outputs_dir": "/content/outputs",
    "data_dir": "/content/eeg_data",
    "kaggle_dataset": "abdelrahmanmahmoud0/eeg-pornography-addiction",
    "gcn": {
        "metric": "coherence",
        "band": "alpha",
        "task": "EC",
        "hidden_dim": 32,
        "max_epochs": 200,
        "patience": 20,
        "lr": 0.001,
        "optuna_trials": 5
    }
}

os.makedirs(CONFIG["outputs_dir"], exist_ok=True)
with open(os.path.join(CONFIG["outputs_dir"], "config.json"), "w") as f:
    json.dump(CONFIG, f, indent=2)

# Seed segala hal
random.seed(CONFIG["random_seed"])
np.random.seed(CONFIG["random_seed"])
torch.manual_seed(CONFIG["random_seed"])
torch.cuda.manual_seed_all(CONFIG["random_seed"])
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
print("CUDA available:", torch.cuda.is_available())
print("Torch version:", torch.__version__)
try:
    import torch_geometric
    print("PyG version:", torch_geometric.__version__)
except Exception as e:
    print("PyG not yet installed (will install below)")



## 1. Install Dependencies

Menyiapkan paket dasar neuroscience + PyTorch Geometric. Install dilakukan sekali di Colab runtime.


In [ ]:

!pip -q install numpy pandas scipy matplotlib mne networkx scikit-learn optuna tqdm
# Install PyTorch (CUDA default di Colab) dan PyG sesuai CUDA
import torch, sys, os, json, subprocess, re

torch_version = torch.__version__.split("+")[0]
cu_version = torch.version.cuda
wheel_url = f"https://data.pyg.org/whl/torch-{torch_version}+cu{cu_version.replace('.', '')}/torch_geometric-2.4.0%2Bcu{cu_version.replace('.', '')}-cp310-cp310-linux_x86_64.whl"
try:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "torch_scatter", "torch_sparse", "torch_cluster", "torch_spline_conv", "-f", wheel_url])
except subprocess.CalledProcessError:
    print("Fallback to CPU wheels")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "torch_scatter", "torch_sparse", "torch_cluster", "torch_spline_conv", "-f", f"https://data.pyg.org/whl/torch-{torch_version}+cpu.html"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "torch_geometric"])

import torch_geometric
print("PyG version after install", torch_geometric.__version__)



## 2. Kaggle Download

Notebook otomatis mengunduh dataset Kaggle. Opsi autentikasi:
1. Upload `kaggle.json` (ke Colab: `Files` → drag & drop). Simpan di `~/.kaggle/kaggle.json`.
2. Atur env `KAGGLE_USERNAME` dan `KAGGLE_KEY` (mis. `os.environ[...] = ...`).


In [ ]:

from pathlib import Path

os.makedirs("/root/.kaggle", exist_ok=True)
if Path("kaggle.json").exists():
    shutil.copy("kaggle.json", "/root/.kaggle/kaggle.json")
    os.chmod("/root/.kaggle/kaggle.json", 0o600)
    print("kaggle.json uploaded and set")

# Fallback ke env var
if not Path("/root/.kaggle/kaggle.json").exists():
    username = os.environ.get("KAGGLE_USERNAME")
    key = os.environ.get("KAGGLE_KEY")
    if username and key:
        with open("/root/.kaggle/kaggle.json", "w") as f:
            json.dump({"username": username, "key": key}, f)
        os.chmod("/root/.kaggle/kaggle.json", 0o600)
        print("kaggle.json created from env vars")
    else:
        raise RuntimeError("Upload kaggle.json atau set env KAGGLE_USERNAME/KAGGLE_KEY")

os.makedirs(CONFIG['data_dir'], exist_ok=True)
!kaggle datasets download -d $CONFIG[kaggle_dataset] -p $CONFIG[data_dir] --force
!unzip -q -o "$CONFIG[data_dir]/*.zip" -d $CONFIG[data_dir]
print("Dataset downloaded to", CONFIG['data_dir'])



## 3. Loading, Channel Mapping, dan Label

- Auto-detect folder `S1..S14` dan file untuk task baseline eyes-closed (EC) dan executive task (ET).
- Normalisasi bentuk data menjadi `[n_samples, n_channels]`.
- Pemetaan kanal mengikuti urutan 19-kanal 10–20. Nama alternatif (T3→T7, T4→T8, T5→P7, T6→P8) otomatis disesuaikan.


In [ ]:

from glob import glob

CHANNEL_MAP = {
    "T3": "T7", "T4": "T8", "T5": "P7", "T6": "P8",
    "AF3": "Fp1", "AF4": "Fp2"  # possible variants
}

LABELS = {f"S{i}": 1 if i <=7 else 0 for i in range(1,15)}

def find_subject_files(root):
    subjects = {}
    for sid in sorted([p for p in Path(root).iterdir() if p.is_dir() and p.name.upper().startswith("S")]):
        files = list(sid.rglob("*"))
        ec = [f for f in files if f.is_file() and "EC" in f.name.upper()]
        et = [f for f in files if f.is_file() and "ET" in f.name.upper()]
        subjects[sid.name] = {"EC": ec, "ET": et}
    return subjects

subjects = find_subject_files(CONFIG['data_dir'])
print("Detected subjects:", list(subjects.keys()))

# Utility loader supporting CSV/TSV/TXT/NPY/EDF

def load_signal(path):
    path = Path(path)
    if path.suffix.lower() in [".npy", ".npz"]:
        arr = np.load(path)
        if isinstance(arr, np.lib.npyio.NpzFile):
            arr = arr[list(arr.files)[0]]
        data = np.array(arr)
        columns = None
    elif path.suffix.lower() in [".csv", ".txt", ".tsv"]:
        df = pd.read_csv(path, sep=None, engine="python")
        data = df.values
        columns = list(df.columns)
    elif path.suffix.lower() in [".edf"]:
        raw = mne.io.read_raw_edf(path, preload=True, verbose=False)
        data = raw.get_data().T  # samples x channels
        columns = raw.ch_names
    else:
        raise ValueError(f"Unsupported format: {path.suffix}")
    if data.shape[0] < data.shape[1]:
        data = data.T
    return data, columns

# Align channels

def reorder_channels(data, columns):
    if columns is None:
        # assume already ordered
        return data[:, :len(CONFIG['channels_order'])]
    cols_std = [CHANNEL_MAP.get(c, c) for c in columns]
    mapping = {c: i for i, c in enumerate(cols_std)}
    ordered = []
    for ch in CONFIG['channels_order']:
        if ch in mapping:
            ordered.append(data[:, mapping[ch]])
        else:
            # pad missing with zeros
            ordered.append(np.zeros(data.shape[0]))
    return np.stack(ordered, axis=1)

summary_rows = []
subject_data = {"EC": {}, "ET": {}}
for sid, tasks in subjects.items():
    for task, files in tasks.items():
        if not files:
            continue
        # pilih file pertama untuk task
        fpath = files[0]
        data_raw, cols = load_signal(fpath)
        data = reorder_channels(data_raw, cols)
        duration = data.shape[0] / CONFIG['sampling_rate']
        summary_rows.append({"subject": sid, "label": LABELS.get(sid, np.nan), "task": task, "duration_sec": duration, "before_epochs": np.nan, "after_epochs": np.nan})
        subject_data[task][sid] = data

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(os.path.join(CONFIG['outputs_dir'], 'summary_initial.csv'), index=False)
summary_df



## 4. Preprocessing & Epoching

- Bandpass 1–50 Hz + notch 50 Hz.
- Artifact rejection berbasis amplitudo ±100 µV per epoch (2 detik, overlap 50%).
- Memilih ~1 menit pertama untuk baseline EC apabila rekaman lebih panjang.
- Band-specific filter alpha (8–13 Hz) dan beta (13–30 Hz) untuk konektivitas.


In [ ]:

from scipy.signal import butter, filtfilt, iirnotch


def butter_bandpass(lowcut, highcut, fs, order=4):
    nyq = 0.5 * fs
    b, a = butter(order, [lowcut / nyq, highcut / nyq], btype='band')
    return b, a

def apply_bandpass(data, low, high, fs, order=4):
    b, a = butter_bandpass(low, high, fs, order)
    return filtfilt(b, a, data, axis=0)

def apply_notch(data, freq, fs, q=30):
    b, a = iirnotch(freq/(fs/2), q)
    return filtfilt(b, a, data, axis=0)

def epoch_data(data, fs, length_sec=2, overlap=0.5):
    step = int(length_sec * fs * (1 - overlap))
    size = int(length_sec * fs)
    epochs = []
    for start in range(0, data.shape[0]-size+1, step):
        epochs.append(data[start:start+size])
    return np.array(epochs)

def reject_artifacts(epochs, threshold=100):
    mask = np.max(np.abs(epochs), axis=(1,2)) < threshold
    return epochs[mask], mask

processed = {"EC": {}, "ET": {}}
summary_rows = []
for task, data_dict in subject_data.items():
    for sid, data in data_dict.items():
        # baseline: ambil 60 detik pertama untuk EC
        if task == "EC":
            max_samples = CONFIG['sampling_rate'] * 60
            data = data[:max_samples]
        data_filt = apply_bandpass(data, CONFIG['bandpass'][0], CONFIG['bandpass'][1], CONFIG['sampling_rate'])
        data_filt = apply_notch(data_filt, CONFIG['notch'], CONFIG['sampling_rate'])
        epochs = epoch_data(data_filt, CONFIG['sampling_rate'], CONFIG['epoch_length_sec'], CONFIG['epoch_overlap'])
        before = len(epochs)
        epochs, mask = reject_artifacts(epochs, CONFIG['artifact_threshold_uV'])
        after = len(epochs)
        processed[task][sid] = epochs
        summary_rows.append({"subject": sid, "label": LABELS[sid], "task": task, "duration_sec": data.shape[0]/CONFIG['sampling_rate'], "before_epochs": before, "after_epochs": after})

summary_df = pd.DataFrame(summary_rows)
summary_path = os.path.join(CONFIG['outputs_dir'], 'summary_epochs.csv')
summary_df.to_csv(summary_path, index=False)
summary_df



## 5. Connectivity: Coherence & PLI

Menghitung matriks konektivitas 19x19 untuk tiap subjek, task (EC, ET), dan band (alpha, beta). Di-cache agar tidak dihitung ulang.


In [ ]:

from functools import lru_cache

os.makedirs(Path(CONFIG['outputs_dir'])/"connectivity", exist_ok=True)

@lru_cache(maxsize=None)
def welch_coherence(x, y, fs, band):
    f, Cxy = signal.coherence(x, y, fs=fs, nperseg=fs*2)
    band_mask = (f >= band[0]) & (f <= band[1])
    return np.mean(Cxy[band_mask])

def compute_coherence_matrix(epochs, fs, band):
    n_channels = epochs.shape[2]
    mat = np.zeros((n_channels, n_channels))
    for i in range(n_channels):
        for j in range(i+1, n_channels):
            vals = [welch_coherence(ep[:, i], ep[:, j], fs, band) for ep in epochs]
            mat[i, j] = mat[j, i] = np.mean(vals)
    np.fill_diagonal(mat, 0)
    return mat

def compute_pli_matrix(epochs, fs, band):
    n_channels = epochs.shape[2]
    mat = np.zeros((n_channels, n_channels))
    for ep in epochs:
        ep_band = apply_bandpass(ep, band[0], band[1], fs)
        analytic = signal.hilbert(ep_band, axis=0)
        phases = np.angle(analytic)
        for i in range(n_channels):
            for j in range(i+1, n_channels):
                dphi = phases[:, i] - phases[:, j]
                pli = np.abs(np.mean(np.sign(np.sin(dphi))))
                mat[i, j] += pli
                mat[j, i] += pli
    mat /= len(epochs)
    np.fill_diagonal(mat, 0)
    return mat

connectivity = {"EC": {}, "ET": {}}
for task, subj_epochs in processed.items():
    for sid, epochs in subj_epochs.items():
        for band_name, band_range in CONFIG['bands'].items():
            for metric in CONFIG['connectivity_metrics']:
                out_dir = Path(CONFIG['outputs_dir'])/"connectivity"/task/metric/band_name
                out_dir.mkdir(parents=True, exist_ok=True)
                out_file = out_dir/f"{sid}.npy"
                if out_file.exists():
                    mat = np.load(out_file)
                else:
                    if metric == "coherence":
                        mat = compute_coherence_matrix(epochs, CONFIG['sampling_rate'], band_range)
                    else:
                        mat = compute_pli_matrix(epochs, CONFIG['sampling_rate'], band_range)
                    np.save(out_file, mat)
                connectivity[task].setdefault(metric, {}).setdefault(band_name, {})[sid] = mat

print("Connectivity computed and cached in", Path(CONFIG['outputs_dir'])/"connectivity")



## 6. Graph Construction (Weighted & Binary)

Threshold binary: buang koneksi di bawah persentil 25% untuk setiap subjek.


In [ ]:

adjacency = {"EC": {}, "ET": {}}
for task in connectivity:
    for metric in connectivity[task]:
        for band in connectivity[task][metric]:
            for sid, mat in connectivity[task][metric][band].items():
                thresh = np.percentile(mat[np.triu_indices_from(mat, k=1)], CONFIG['threshold_percentile'])
                binary = (mat >= thresh).astype(float)
                np.fill_diagonal(binary, 0)
                out_dir = Path(CONFIG['outputs_dir'])/"adjacency"/task/metric/band
                out_dir.mkdir(parents=True, exist_ok=True)
                np.save(out_dir/f"{sid}_weighted.npy", mat)
                np.save(out_dir/f"{sid}_binary.npy", binary)
                adjacency[task].setdefault(metric, {}).setdefault(band, {})[sid] = {"weighted": mat, "binary": binary}
print("Adjacency matrices stored under outputs/adjacency")



## 7. Graph-Theoretic Metrics & Statistik

Menghitung metrik: degree, clustering, path length (komponen terbesar), global efficiency, small-world index (dibanding graf acak Erdos-Renyi dengan densitas sama), dan betweenness. Uji Mann–Whitney U (alpha=0.05) + effect size rank-biserial.


In [ ]:

def graph_metrics(adj):
    G = nx.from_numpy_array(adj)
    if not nx.is_connected(G):
        Gc = G.subgraph(max(nx.connected_components(G), key=len)).copy()
    else:
        Gc = G
    deg = np.array(list(dict(G.degree(weight='weight')).values()))
    clustering = nx.average_clustering(G, weight='weight')
    try:
        path_length = nx.average_shortest_path_length(Gc, weight=lambda u,v,d: 1/max(d.get('weight',1e-6),1e-6))
    except nx.NetworkXError:
        path_length = np.nan
    eff = nx.global_efficiency(G)
    bc = np.mean(list(nx.betweenness_centrality(G, weight='weight').values()))
    # small-world sigma
    n = G.number_of_nodes()
    p = np.count_nonzero(adj) / (n*(n-1))
    Gr = nx.erdos_renyi_graph(n, p)
    sigma = (nx.average_clustering(G)/nx.average_clustering(Gr)) / (nx.average_shortest_path_length(Gc)/nx.average_shortest_path_length(Gr))
    return {
        "degree_mean": deg.mean(),
        "clustering": clustering,
        "path_length": path_length,
        "efficiency": eff,
        "betweenness": bc,
        "sigma": sigma
    }

metrics_rows = []
for task in adjacency:
    for metric in adjacency[task]:
        for band in adjacency[task][metric]:
            for sid, mats in adjacency[task][metric][band].items():
                row = {"subject": sid, "label": LABELS[sid], "task": task, "metric": metric, "band": band}
                row.update(graph_metrics(mats['weighted']))
                metrics_rows.append(row)
metrics_df = pd.DataFrame(metrics_rows)
metrics_path = Path(CONFIG['outputs_dir'])/"graph_metrics.csv"
metrics_df.to_csv(metrics_path, index=False)
metrics_df.head()


In [ ]:

# Statistik Mann-Whitney U per kombinasi
stats_rows = []
for task in metrics_df['task'].unique():
    for metric in metrics_df['metric'].unique():
        for band in metrics_df['band'].unique():
            subset = metrics_df[(metrics_df.task==task)&(metrics_df.metric==metric)&(metrics_df.band==band)]
            for col in ['degree_mean','clustering','path_length','efficiency','betweenness','sigma']:
                g1 = subset[subset.label==1][col]
                g0 = subset[subset.label==0][col]
                if len(g1)==0 or len(g0)==0:
                    continue
                stat, p = mannwhitneyu(g1, g0, alternative='two-sided')
                effect = (g1.mean() - g0.mean()) / subset[col].std(ddof=1)
                stats_rows.append({"task": task, "metric": metric, "band": band, "feature": col, "ppu_mean": g1.mean(), "control_mean": g0.mean(), "p_value": p, "effect_size": effect})
stats_df = pd.DataFrame(stats_rows)
stats_path = Path(CONFIG['outputs_dir'])/"graph_stats.csv"
stats_df.to_csv(stats_path, index=False)
stats_df.head()



## 8. Visualisasi

- Heatmap konektivitas rata-rata grup (PPU vs Control) untuk EC/ET dan metric/band terpilih.
- Plot graf pada layout scalp (montage standard 10–20).
- Distribusi metrik graf per grup dengan p-value.


In [ ]:

import matplotlib.pyplot as plt
import seaborn as sns

fig_dir = Path(CONFIG['outputs_dir'])/"figures"
fig_dir.mkdir(parents=True, exist_ok=True)

montage = mne.channels.make_standard_montage('standard_1020')
pos = {ch: montage.get_positions()['ch_pos'][ch][:2] for ch in CONFIG['channels_order']}

# Heatmaps
for task in ['EC','ET']:
    for metric in ['coherence','pli']:
        for band in ['alpha','beta']:
            mats_ppu = [connectivity[task][metric][band][sid] for sid in connectivity[task][metric][band] if LABELS[sid]==1]
            mats_ctrl = [connectivity[task][metric][band][sid] for sid in connectivity[task][metric][band] if LABELS[sid]==0]
            if not mats_ppu or not mats_ctrl:
                continue
            mean_ppu = np.mean(mats_ppu, axis=0)
            mean_ctrl = np.mean(mats_ctrl, axis=0)
            for grp, mat in [("PPU", mean_ppu), ("Control", mean_ctrl)]:
                plt.figure(figsize=(8,6))
                sns.heatmap(mat, vmin=0, vmax=np.max(mat), cmap='viridis')
                plt.title(f"{grp} {task} {metric} {band}")
                out = fig_dir/f"heatmap_{grp}_{task}_{metric}_{band}.png"
                plt.savefig(out, dpi=150, bbox_inches='tight')
                plt.close()

# Graph layout visualization using weighted adjacency from average PPU
for task in ['EC','ET']:
    for metric in ['coherence','pli']:
        for band in ['alpha','beta']:
            mats = [connectivity[task][metric][band][sid] for sid in connectivity[task][metric][band]]
            if not mats:
                continue
            mean_mat = np.mean(mats, axis=0)
            thresh = np.percentile(mean_mat[np.triu_indices_from(mean_mat, k=1)], CONFIG['threshold_percentile'])
            G = nx.from_numpy_array(mean_mat)
            edges = [(u,v,d['weight']) for u,v,d in G.edges(data=True) if d['weight']>=thresh]
            plt.figure(figsize=(6,6))
            for u,v,w in edges:
                x=[pos[CONFIG['channels_order'][u]][0], pos[CONFIG['channels_order'][v]][0]]
                y=[pos[CONFIG['channels_order'][u]][1], pos[CONFIG['channels_order'][v]][1]]
                plt.plot(x, y, color='C0', alpha=0.6, linewidth=2*w)
            for i,ch in enumerate(CONFIG['channels_order']):
                plt.scatter(pos[ch][0], pos[ch][1], color='k')
                plt.text(pos[ch][0], pos[ch][1], ch, fontsize=8)
            plt.axis('off')
            plt.title(f"Graph {task} {metric} {band}")
            out = fig_dir/f"graph_{task}_{metric}_{band}.png"
            plt.savefig(out, dpi=150, bbox_inches='tight')
            plt.close()

# Boxplots for metrics
for feat in ['degree_mean','clustering','path_length','efficiency','betweenness','sigma']:
    plt.figure(figsize=(8,4))
    sns.boxplot(data=metrics_df, x='label', y=feat, hue='task')
    plt.xticks([0,1],["Control","PPU"])
    plt.title(f"Distribution {feat}")
    out = fig_dir/f"box_{feat}.png"
    plt.savefig(out, dpi=150, bbox_inches='tight')
    plt.close()



## 9. PyTorch Geometric Dataset Builder

Node feature: rata-rata band power alpha per kanal. Edge: weighted adjacency (metric/band/task sesuai CONFIG). Edge attr = weight. Label: PPU=1, Control=0.


In [ ]:

from torch_geometric.data import Data
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


def bandpower(signal_data, fs, band):
    f, Pxx = signal.welch(signal_data, fs=fs, nperseg=fs*2, axis=0)
    mask = (f>=band[0])&(f<=band[1])
    return np.trapz(Pxx[mask], f[mask], axis=0)

selected_metric = CONFIG['gcn']['metric']
selected_band = CONFIG['gcn']['band']
selected_task = CONFIG['gcn']['task']

pyg_graphs = []
for sid in connectivity[selected_task][selected_metric][selected_band]:
    mat = connectivity[selected_task][selected_metric][selected_band][sid]
    src, dst = np.where(mat > 0)
    weights = mat[src, dst]
    edge_index = torch.tensor(np.vstack([src, dst]), dtype=torch.long)
    edge_attr = torch.tensor(weights[:, None], dtype=torch.float32)
    # node feature band power from task epochs
    epochs = processed[selected_task][sid]
    bp = []
    for ep in epochs:
        bp.append(bandpower(ep, CONFIG['sampling_rate'], CONFIG['bands']['alpha']))
    bp = np.mean(bp, axis=0)
    x = torch.tensor(bp[:, None], dtype=torch.float32)
    y = torch.tensor([LABELS[sid]], dtype=torch.long)
    pyg_graphs.append((sid, Data(x=x, edge_index=edge_index, edge_attr=edge_attr, y=y)))

print(f"Built {len(pyg_graphs)} graphs for task {selected_task} metric {selected_metric} band {selected_band}")



## 10. GCN Model + LOOCV (Optuna)

- 2 layer GCNConv + ReLU + global mean pooling.
- Optuna untuk mencari hidden_dim, dropout, dan lr (jumlah trial diset kecil agar cepat, dapat dinaikkan).
- LOOCV 14 subjek dengan split train/val (90/10) di dalam fold.
- Anti-leakage: scaler fit pada data train fold saja.


In [ ]:

from torch_geometric.nn import GCNConv, global_mean_pool
import optuna

class GCN(nn.Module):
    def __init__(self, in_dim, hidden_dim, dropout=0.2):
        super().__init__()
        self.conv1 = GCNConv(in_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim)
        self.lin = nn.Linear(hidden_dim, 2)
        self.dropout = dropout
    def forward(self, data):
        x, edge_index, edge_attr, batch = data.x, data.edge_index, data.edge_attr, data.batch
        x = self.conv1(x, edge_index, edge_weight=edge_attr.squeeze())
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.conv2(x, edge_index, edge_weight=edge_attr.squeeze())
        x = F.relu(x)
        x = global_mean_pool(x, batch)
        return self.lin(x)


def train_model(model, optimizer, criterion, loader_train, loader_val, patience=20, max_epochs=200):
    best_loss = np.inf
    best_state = None
    no_improve = 0
    for epoch in range(max_epochs):
        model.train()
        for batch in loader_train:
            batch = batch.to(device)
            optimizer.zero_grad()
            out = model(batch)
            loss = criterion(out, batch.y)
            loss.backward()
            optimizer.step()
        # val
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for batch in loader_val:
                batch = batch.to(device)
                out = model(batch)
                val_loss += criterion(out, batch.y).item()
        val_loss /= max(1, len(loader_val))
        if val_loss < best_loss:
            best_loss = val_loss
            best_state = model.state_dict()
            no_improve = 0
        else:
            no_improve +=1
        if no_improve >= patience:
            break
    if best_state:
        model.load_state_dict(best_state)
    return model

from torch_geometric.loader import DataLoader
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

results = []

# Optuna objective

def objective(trial, train_graphs, val_graphs):
    hidden = trial.suggest_int('hidden_dim', 16, 64, step=16)
    dropout = trial.suggest_float('dropout', 0.1, 0.5)
    lr = trial.suggest_float('lr', 1e-4, 5e-3, log=True)
    model = GCN(1, hidden, dropout).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    loader_train = DataLoader([g for _,g in train_graphs], batch_size=4, shuffle=True)
    loader_val = DataLoader([g for _,g in val_graphs], batch_size=4, shuffle=False)
    model = train_model(model, optimizer, criterion, loader_train, loader_val, patience=CONFIG['gcn']['patience'], max_epochs=CONFIG['gcn']['max_epochs'])
    model.eval()
    preds, labels = [], []
    with torch.no_grad():
        for batch in loader_val:
            batch = batch.to(device)
            out = model(batch)
            preds.extend(out.argmax(dim=1).cpu().numpy())
            labels.extend(batch.y.cpu().numpy())
    return accuracy_score(labels, preds)

fold_reports = []
for test_idx in range(len(pyg_graphs)):
    test_sid, test_graph = pyg_graphs[test_idx]
    train_graphs = [g for i,g in enumerate(pyg_graphs) if i!=test_idx]
    # split train/val
    ids = list(range(len(train_graphs)))
    random.shuffle(ids)
    split = int(len(ids)*0.9)
    train_ids, val_ids = ids[:split], ids[split:]
    train_set = [train_graphs[i] for i in train_ids]
    val_set = [train_graphs[i] for i in val_ids]

    study = optuna.create_study(direction='maximize')
    study.optimize(lambda trial: objective(trial, train_set, val_set), n_trials=CONFIG['gcn']['optuna_trials'], show_progress_bar=False)
    best_params = study.best_params

    model = GCN(1, best_params['hidden_dim'], best_params['dropout']).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=best_params['lr'])
    criterion = nn.CrossEntropyLoss()
    loader_train = DataLoader([g for _,g in train_set+val_set], batch_size=4, shuffle=True)
    loader_test = DataLoader([test_graph[1]], batch_size=1)
    model = train_model(model, optimizer, criterion, loader_train, loader_test, patience=CONFIG['gcn']['patience'], max_epochs=CONFIG['gcn']['max_epochs'])
    model.eval()
    with torch.no_grad():
        for batch in loader_test:
            batch = batch.to(device)
            pred = model(batch).argmax(dim=1).item()
            true = batch.y.item()
            results.append({"test_subject": test_sid, "true": true, "pred": pred, "params": best_params})

res_df = pd.DataFrame(results)
acc = accuracy_score(res_df['true'], res_df['pred'])
cm = confusion_matrix(res_df['true'], res_df['pred'])
report = classification_report(res_df['true'], res_df['pred'], target_names=['Control','PPU'], output_dict=True)

print("LOOCV accuracy", acc)
print("Confusion matrix
", cm)

res_df.to_csv(Path(CONFIG['outputs_dir'])/"gcn_predictions.csv", index=False)
pd.DataFrame(report).to_csv(Path(CONFIG['outputs_dir'])/"gcn_classification_report.csv")



## 11. Interpretability & Ablation

1. **Frontal ablation**: hilangkan edge yang incident ke Fp1/Fp2 (opsional tambah F3/F4/Fz) lalu evaluasi ulang.
2. **Embedding PCA**: ambil embedding node layer terakhir dan lakukan PCA 2D.
3. **Edge importance**: ranking bobot edge terbesar.


In [ ]:

from sklearn.decomposition import PCA

# gunakan model terlatih terakhir
model.eval()
frontal_nodes = [CONFIG['channels_order'].index(ch) for ch in ['Fp1','Fp2','F3','F4','Fz']]

# Ablation pada graf test terakhir
ablation_results = []
for sid, data in pyg_graphs:
    mat = connectivity[selected_task][selected_metric][selected_band][sid].copy()
    for fn in frontal_nodes:
        mat[fn,:] = 0; mat[:,fn]=0
    src, dst = np.where(mat>0)
    edge_attr = torch.tensor(mat[src,dst][:,None], dtype=torch.float32)
    edge_index = torch.tensor(np.vstack([src,dst]), dtype=torch.long)
    x = data.x
    batch = torch.zeros(x.size(0), dtype=torch.long)
    g = Data(x=x, edge_index=edge_index, edge_attr=edge_attr, y=data.y, batch=batch)
    with torch.no_grad():
        out = model(g.to(device))
        prob = F.softmax(out, dim=1).cpu().numpy()[0]
        ablation_results.append({"subject": sid, "prob_control": prob[0], "prob_ppu": prob[1]})

ablation_df = pd.DataFrame(ablation_results)
ablation_df.to_csv(Path(CONFIG['outputs_dir'])/"ablation_frontal.csv", index=False)
ablation_df.head()

# Embedding PCA
embeddings = []
labels = []
with torch.no_grad():
    for sid, data in pyg_graphs:
        batch = torch.zeros(data.x.size(0), dtype=torch.long)
        data.batch = batch
        data = data.to(device)
        x = model.conv1(data.x, data.edge_index, edge_weight=data.edge_attr.squeeze())
        x = F.relu(x)
        x = model.conv2(x, data.edge_index, edge_weight=data.edge_attr.squeeze())
        embeddings.append(x.cpu().numpy())
        labels.append(LABELS[sid])
all_emb = np.vstack(embeddings)
pca = PCA(n_components=2)
proj = pca.fit_transform(all_emb)
labels_rep = np.concatenate([[lbl]*config.shape[0] for lbl,config in zip(labels, embeddings)])

plt.figure(figsize=(6,5))
plt.scatter(proj[labels_rep==0,0], proj[labels_rep==0,1], label='Control', alpha=0.6)
plt.scatter(proj[labels_rep==1,0], proj[labels_rep==1,1], label='PPU', alpha=0.6)
plt.legend(); plt.title('Node embedding PCA')
pca_path = fig_dir/"embedding_pca.png"
plt.savefig(pca_path, dpi=150, bbox_inches='tight')
plt.close()

# Edge importance (top weights)
importance_rows = []
for sid, data in pyg_graphs:
    mat = connectivity[selected_task][selected_metric][selected_band][sid]
    idx = np.triu_indices_from(mat, k=1)
    weights = mat[idx]
    top_k = np.argsort(weights)[-10:]
    for k in top_k:
        ch1 = CONFIG['channels_order'][idx[0][k]]
        ch2 = CONFIG['channels_order'][idx[1][k]]
        importance_rows.append({"subject": sid, "edge": f"{ch1}-{ch2}", "weight": weights[k]})
imp_df = pd.DataFrame(importance_rows).sort_values("weight", ascending=False)
imp_df.to_csv(Path(CONFIG['outputs_dir'])/"edge_importance.csv", index=False)
imp_df.head()



## 12. Export Outputs

Zip seluruh konten `/content/outputs` agar mudah diunduh dari Colab.


In [ ]:

zip_path = "/content/outputs_all.zip"
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, _, files in os.walk(CONFIG['outputs_dir']):
        for file in files:
            abs_path = os.path.join(root, file)
            rel = os.path.relpath(abs_path, '/content')
            zf.write(abs_path, rel)
print("Zipped outputs at", zip_path)
